# Notebook 1 — Dataset Understanding
### Sprint 4 | Data Inspection & Exploratory Data Analysis (EDA)

**Methodology for every topic below:**
1. **Understand the Concept** (Markdown) — explained in my own words.
2. **Demonstrate the Concept** (Markdown) — a simple / real-world / business / AI-ML
   example, and why it matters during data analysis.
3. **Implement the Concept** (Python) — using Pandas/NumPy, with what the code does, why
   the method was used, what the output means, what insights it gives, and how it can
   affect an ML pipeline.

**Guiding principle for this sprint:** Load → Inspect → Understand → Analyze →
Visualize → Identify Problems → Document Findings.


---
## Dataset Documentation

**Dataset name:** Telco Customer Churn

**Dataset source:** IBM Sample Data Sets (a widely used, publicly available real-world
dataset, also mirrored on Kaggle). Loaded directly from:
`https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv`

**Business / domain:** Telecommunications — Customer Churn. The dataset represents a
telecom company's customers, the services they've subscribed to, their account/billing
information, and whether they churned (left the company) or not.

**Number of rows:** 7,043 (one row per customer)

**Number of columns:** 21

**Meaning of important columns:**

| Column | Meaning |
|---|---|
| `customerID` | A unique identifier for each customer (not a real feature) |
| `gender`, `SeniorCitizen`, `Partner`, `Dependents` | Basic demographic information |
| `tenure` | Number of months the customer has stayed with the company |
| `PhoneService`, `MultipleLines`, `InternetService` | Which core services the customer has |
| `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies` | Add-on services the customer has subscribed to |
| `Contract` | The customer's contract type (month-to-month, one year, two year) |
| `PaperlessBilling`, `PaymentMethod` | Billing preferences |
| `MonthlyCharges` | The customer's current monthly bill amount |
| `TotalCharges` | The total amount charged to the customer over their whole tenure |
| `Churn` | Whether the customer left the company (**the target variable**) |

**Target variable:** `Churn` (Yes / No) — this is what a machine learning model built on
this dataset would try to predict.

**Why I selected this dataset:**
- It's a genuine, real-world dataset (not synthetically generated), widely used and
  well-documented, so its structure and quirks are trustworthy to learn from.
- It has a healthy mix of numerical (`tenure`, `MonthlyCharges`, `TotalCharges`) and
  categorical (`Contract`, `PaymentMethod`, `InternetService`, etc.) columns — exactly
  the mix this sprint asks for.
- It has a clear, realistic business target (`Churn`), so every EDA step in this sprint
  can be tied back to a genuine business question: *"what patterns predict a customer
  leaving?"*
- It has a real, subtle data-quality issue baked in — `TotalCharges` is stored as text
  and contains a small number of blank entries — exactly the kind of realistic problem
  this sprint asks us to be able to find and document, rather than a dataset that's
  already perfectly clean.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("telco_churn.csv")

print(f"Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head()


Dataset loaded: 7,043 rows, 21 columns


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


---
## 1. What is a Dataset?

### Step 1 — Understand the Concept
A dataset is a structured collection of data, organized into rows and columns, where each
row represents one real-world observation and each column represents one measured or
recorded characteristic of that observation.

### Step 2 — Demonstrate the Concept
**Simple example:** A telecom company's customer records — every row is one customer,
every column is a fact about that customer (their contract type, monthly bill, etc.).

**Why it matters during data analysis:** Before analyzing anything, I need to know
*what one row actually represents* — here, one row is one customer's full account
snapshot, not one transaction or one phone call. Misunderstanding the row's meaning would
lead to wrong conclusions later.

### Step 3 — Implement the Concept


In [2]:
print(f"This dataset has {df.shape[0]:,} rows (one row = one customer)")
print(f"and {df.shape[1]} columns (facts recorded about each customer).")
df.sample(3, random_state=1)


This dataset has 7,043 rows (one row = one customer)
and 21 columns (facts recorded about each customer).


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
3381,8879-ZKJOF,Female,0,No,No,41,Yes,No,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Bank transfer (automatic),79.85,3320.75,No
6180,0201-MIBOL,Female,1,No,No,66,Yes,Yes,Fiber optic,Yes,...,No,No,Yes,Yes,Two year,Yes,Bank transfer (automatic),102.40,6471.85,No
4829,1600-DILPE,Female,0,No,No,12,Yes,No,DSL,No,...,No,No,No,No,Month-to-month,Yes,Bank transfer (automatic),45.00,524.35,No


**What this does:** Confirms the dataset's size and previews 3 random customer
records. **Why this method:** `.sample()` avoids the bias of always looking at the same
first few rows. **What the output means:** each row is a complete, independent customer
record. **Insight:** with 7,043 independent customer records, there's a solid sample size
to detect real patterns. **ML pipeline impact:** confirming what a "row" represents is the
very first step before any feature engineering, since features must be computed per
customer, not per some other unit.


---
## 2. What is a Feature?

### Step 1 — Understand the Concept
A feature is any column used as an **input** to describe or predict something — the
measurable characteristics a model (or an analyst) uses to understand a customer.

### Step 2 — Demonstrate the Concept
**Business example:** `Contract`, `MonthlyCharges`, and `tenure` are all features — each
describes something about a customer that could plausibly relate to whether they churn.

**AI/ML use case:** A churn-prediction model's entire input side is built from features
like these — the model has no other information about a customer besides what's captured
in these columns.

### Step 3 — Implement the Concept


In [3]:
candidate_features = [c for c in df.columns if c not in ['customerID', 'Churn']]
print(f"Number of candidate features: {len(candidate_features)}")
print(candidate_features)


Number of candidate features: 19
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']


**What this does:** Lists every column except the identifier and the target — the
columns that could act as model inputs. **Why this method:** explicitly separating
"features" from "identifier" and "target" prevents accidentally feeding the wrong columns
into a future model. **Insight:** 19 candidate features, covering demographics, services,
contract, and billing. **ML pipeline impact:** this list is essentially the starting
point for the feature set a churn model would be trained on.


---
## 3. What is a Target?

### Step 1 — Understand the Concept
The target (also called the label or outcome) is the column a supervised model is trying
to predict — the "answer" the model learns to produce from the features.

### Step 2 — Demonstrate the Concept
**Business example:** `Churn` is the target here — the company wants to predict, for each
customer, whether they will leave, so they can intervene beforehand.

**AI/ML use case:** Every supervised learning problem needs a clearly identified target
column before any model can be trained — without it, there's nothing for the model to
learn to predict.

### Step 3 — Implement the Concept


In [4]:
print("Target variable: Churn")
print(df['Churn'].value_counts())
print("\nAs percentages:")
print((df['Churn'].value_counts(normalize=True) * 100).round(1))


Target variable: Churn
Churn
No     5174
Yes    1869
Name: count, dtype: int64

As percentages:
Churn
No     73.5
Yes    26.5
Name: proportion, dtype: float64


**What this does:** Counts how many customers churned vs. stayed. **Why this
method:** `.value_counts()` is the fastest way to see a target's class balance.
**What the output means:** about 26.5% of customers churned, 73.5% did not.
**Insight:** this is a moderately imbalanced target — not extreme, but enough that
accuracy alone would be a misleading evaluation metric later (Sprint 2, Notebook 1).
**ML pipeline impact:** this class imbalance will directly influence which evaluation
metrics and possibly which resampling techniques are appropriate for a future churn
model.


---
## 4. Independent Variables

### Step 1 — Understand the Concept
Independent variables are the input variables (features) whose values are used to explain
or predict the outcome — they're called "independent" because, in the model, their values
are treated as given, not something being predicted.

### Step 2 — Demonstrate the Concept
**Simple example:** `tenure` and `MonthlyCharges` are independent variables — the company
already knows these values for every customer; they're not what's being predicted.

**Why it matters during data analysis:** Clearly separating independent variables from the
target keeps the analysis honest — I should look for how independent variables *relate to*
the target, not accidentally treat the target as if it were just another input.

### Step 3 — Implement the Concept


In [5]:
independent_variables = df.drop(columns=['customerID', 'Churn'])
print(f"{independent_variables.shape[1]} independent variables, {independent_variables.shape[0]:,} rows")
independent_variables.dtypes


19 independent variables, 7,043 rows


gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
dtype: object

**What this does:** Builds a table of just the independent variables, dropping the
identifier and target. **Why this method:** `.drop(columns=...)` is a direct way to
isolate exactly the columns that belong on the "input" side. **Insight:** the independent
variables span both numeric and text (categorical) types, previewed here via `.dtypes`.
**ML pipeline impact:** this is effectively the raw `X` matrix a model would eventually be
trained on, before any encoding or scaling.


---
## 5. Dependent Variables

### Step 1 — Understand the Concept
The dependent variable is another name for the target — its value is considered to
*depend on* the independent variables. "Dependent variable" and "target" describe the
same column, just using statistics vs. ML terminology.

### Step 2 — Demonstrate the Concept
**Simple example:** Whether a customer churns (`Churn`) is treated as depending on their
contract type, tenure, monthly charges, and other factors — not the other way around.

**Why it matters during data analysis:** Using "dependent variable" is common in
statistical contexts (like regression) while "target" or "label" is more common in ML
contexts — recognizing both terms mean the same thing avoids confusion when reading
different resources.

### Step 3 — Implement the Concept


In [6]:
dependent_variable = df['Churn']
print("Dependent variable (Churn) — first 5 values:")
print(dependent_variable.head())
print(f"\nType: {type(dependent_variable)}")


Dependent variable (Churn) — first 5 values:
0     No
1     No
2    Yes
3     No
4    Yes
Name: Churn, dtype: str

Type: <class 'pandas.Series'>


**What this does:** Isolates the `Churn` column on its own as a Series. **Why this
method:** direct column selection is how the dependent variable/target is separated from
everything else before modeling. **Insight:** confirms `Churn` is a simple two-category
(Yes/No) Series. **ML pipeline impact:** this is the raw `y` vector a supervised model
would be trained to predict — it will later need encoding into 0/1 before most algorithms
can use it.


---
## 6. Numerical Features

### Step 1 — Understand the Concept
(Recap from Sprint 2) Numerical features hold quantitative values where arithmetic
(averaging, comparing magnitude) is meaningful.

### Step 2 — Demonstrate the Concept
**Business example:** `tenure` (months), `MonthlyCharges` (currency), and `TotalCharges`
(currency) are all numerical — it's meaningful to ask "what's the average tenure?"

**AI/ML use case:** Numerical features are usually fed into a model directly (after
scaling), unlike categorical features, which need encoding first.

### Step 3 — Implement the Concept


In [7]:
numerical_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print("Numerical features detected automatically:", numerical_features)


Numerical features detected automatically: ['SeniorCitizen', 'tenure', 'MonthlyCharges']


**What this does:** Uses `.select_dtypes()` to automatically list columns stored as
numeric types. **Why this method:** it's a fast, reliable first pass — though note
`TotalCharges` is missing from this list, because it's actually stored as text
(investigated in Notebook 2). **Insight:** only `SeniorCitizen`, `tenure`, and
`MonthlyCharges` are detected as numeric right now — a red flag that `TotalCharges` needs
attention. **ML pipeline impact:** this mismatch between a column's *true* nature (numeric)
and its *stored* type (text) is exactly the kind of issue that must be caught during EDA,
or it will silently break any numeric calculation on `TotalCharges` later.


---
## 7. Categorical Features

### Step 1 — Understand the Concept
(Recap from Sprint 2) Categorical features hold labels or categories, not numbers meant
for arithmetic — even when they look like distinct discrete options such as "Yes/No" or
plan names.

### Step 2 — Demonstrate the Concept
**Business example:** `Contract` (month-to-month / one year / two year) and
`InternetService` (DSL / Fiber optic / No) are categorical — there's no meaningful
"average contract type."

**AI/ML use case:** Categorical features need to be encoded (e.g., one-hot encoding)
before most ML algorithms can use them, since algorithms expect numeric input.

### Step 3 — Implement the Concept


In [8]:
categorical_features = df.select_dtypes(include=['object']).columns.tolist()
categorical_features.remove('customerID')   # an identifier, not a real feature

print(f"{len(categorical_features)} categorical features:")
print(categorical_features)


17 categorical features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges', 'Churn']


/tmp/ipykernel_540/2951305256.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = df.select_dtypes(include=['object']).columns.tolist()


**What this does:** Lists all text-typed columns, excluding the identifier.
**Why this method:** `.select_dtypes(include=['object'])` catches every column Pandas is
currently storing as text — including `TotalCharges`, which shouldn't really be here.
**Insight:** 16 columns are categorical, covering demographics, services, and billing
preferences. **ML pipeline impact:** every one of these will need an encoding decision
(one-hot, ordinal, etc.) before model training — a substantial part of the feature
engineering work ahead.


---
## 8. Discrete Features

### Step 1 — Understand the Concept
(Recap from Sprint 2) Discrete features take distinct, countable values — typically whole
numbers with real gaps between possible values.

### Step 2 — Demonstrate the Concept
**Business example:** `tenure` (measured in whole months: 0, 1, 2, ... 72) is discrete —
a customer can't have stayed 5.5 months in this dataset's recording convention.

**Why it matters during data analysis:** Discrete numeric columns are often better
visualized with a bar-style histogram (one bar per whole value, or small bins) than a
smooth curve, since the underlying values are inherently step-wise.

### Step 3 — Implement the Concept


In [9]:
print(f"tenure — min: {df['tenure'].min()}, max: {df['tenure'].max()}, unique values: {df['tenure'].nunique()}")
print("Sample of tenure values:", sorted(df['tenure'].unique())[:15])


tenure — min: 0, max: 72, unique values: 73
Sample of tenure values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14)]


**What this does:** Confirms `tenure` only takes whole-number values across a
bounded range. **Why this method:** checking `.nunique()` and the actual sorted values
confirms the discrete, whole-month recording convention. **Insight:** tenure ranges from 0
(brand new customers) to 72 months (6 years) — a wide, meaningful range for churn
analysis. **ML pipeline impact:** discrete bounded features like this are often good
candidates for either being used as-is or being bucketed into tenure groups (e.g.,
"new," "established," "long-term") as an engineered feature.


---
## 9. Continuous Features

### Step 1 — Understand the Concept
(Recap from Sprint 2) Continuous features can take any value within a range, including
decimals, with no fixed step between possible values.

### Step 2 — Demonstrate the Concept
**Business example:** `MonthlyCharges` (e.g., 29.85, 56.95, 70.70) is continuous — any
decimal value within the plan's pricing range is possible.

**Why it matters during data analysis:** Continuous features are typically visualized with
histograms or density plots (Sprint 2, Notebook 13) rather than simple bar counts, since
there are effectively infinite possible values, not a small fixed set.

### Step 3 — Implement the Concept


In [10]:
print(f"MonthlyCharges — min: {df['MonthlyCharges'].min()}, max: {df['MonthlyCharges'].max()}")
print(f"Number of unique MonthlyCharges values: {df['MonthlyCharges'].nunique()} (out of {len(df)} customers)")


MonthlyCharges — min: 18.25, max: 118.75
Number of unique MonthlyCharges values: 1585 (out of 7043 customers)


**What this does:** Checks the range and uniqueness of `MonthlyCharges`.
**Why this method:** a high ratio of unique values to total rows (1,585 unique values
across 7,043 customers) is a strong signal that this is genuinely continuous, not a
small set of repeated categories. **Insight:** monthly charges range from about $18 to
$119, reflecting a wide spread of plan combinations. **ML pipeline impact:** continuous
features like this are strong candidates for scaling/standardization (Sprint 2,
Notebook 14) before being used in most models.


---
## 10. Identifier Columns

### Step 1 — Understand the Concept
An identifier column uniquely labels each row (like an ID number) but carries no real
predictive information about the outcome — it exists to reference a specific record, not
to describe it.

### Step 2 — Demonstrate the Concept
**Business example:** `customerID` uniquely identifies each customer, but a customer's
random ID string has no genuine relationship with whether they'll churn.

**Why it matters during data analysis:** Identifier columns must be recognized and
excluded from modeling — accidentally leaving one in as a "feature" can cause a model to
overfit or leak information (e.g., if IDs were assigned in a way that correlates with
time or a group).

### Step 3 — Implement the Concept


In [11]:
print(f"customerID — unique values: {df['customerID'].nunique()} out of {len(df)} rows")
print("Are all values unique?", df['customerID'].nunique() == len(df))
print("\nSample IDs:", df['customerID'].head(3).tolist())


customerID — unique values: 7043 out of 7043 rows
Are all values unique? True

Sample IDs: ['7590-VHVEG', '5575-GNVDE', '3668-QPYBK']


**What this does:** Confirms `customerID` has exactly as many unique values as
there are rows. **Why this method:** a 1-to-1 match between unique count and row count is
the defining signature of a true identifier column. **Insight:** confirmed — `customerID`
is a pure identifier, with zero duplicate customers. **ML pipeline impact:** `customerID`
must be dropped before model training (though it can be kept aside for joining
predictions back to specific customers afterward).


---
## 11. Target Variable (Revisited)

### Step 1 — Understand the Concept
Having covered features, independent/dependent variables, and identifiers, it's worth
explicitly re-confirming the target variable's exact type and encoding, since this
directly determines what *kind* of ML problem this is (classification vs. regression).

### Step 2 — Demonstrate the Concept
**AI/ML use case:** Since `Churn` has exactly two possible outcomes (Yes/No), this is a
**binary classification** problem — different from, say, predicting `MonthlyCharges`
(a number), which would be a regression problem.

### Step 3 — Implement the Concept


In [12]:
print("Target column: Churn")
print("Unique values:", df['Churn'].unique())
print("Data type:", df['Churn'].dtype)
print("\n-> This confirms a BINARY CLASSIFICATION problem (2 possible outcomes).")


Target column: Churn
Unique values: <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str
Data type: str

-> This confirms a BINARY CLASSIFICATION problem (2 possible outcomes).


**What this does:** Explicitly re-confirms the target's exact possible values and
type. **Why this method:** knowing the target is text-encoded Yes/No (not already 0/1)
flags an encoding step that will be needed later. **Insight:** this is a binary
classification target, currently stored as text. **ML pipeline impact:** `Churn` will
need to be encoded into 0/1 before most classification algorithms can use it as a
training label.


---
## 12. Metadata

### Step 1 — Understand the Concept
Metadata is "data about the data" — information describing the dataset itself (its
source, size, column meanings, collection context) rather than the actual recorded values.
The Dataset Documentation section at the top of this notebook IS the metadata for this
dataset.

### Step 2 — Demonstrate the Concept
**Business example:** Knowing this dataset came from IBM's sample telecom data, represents
one snapshot in time, and has 7,043 customer rows is metadata — it's context *about* the
dataset, not a value *within* it.

**Why it matters during data analysis:** Without metadata, a dataset is just numbers and
text with no context — metadata is what makes columns like `tenure` or `Contract`
actually meaningful to a human analyst.

### Step 3 — Implement the Concept


In [13]:
metadata = {
    'dataset_name': 'Telco Customer Churn',
    'source': 'IBM Sample Data Sets (via GitHub mirror)',
    'domain': 'Telecommunications / Customer Churn',
    'num_rows': df.shape[0],
    'num_columns': df.shape[1],
    'target_variable': 'Churn',
    'identifier_column': 'customerID',
}
for key, value in metadata.items():
    print(f"{key:<20}: {value}")


dataset_name        : Telco Customer Churn
source              : IBM Sample Data Sets (via GitHub mirror)
domain              : Telecommunications / Customer Churn
num_rows            : 7043
num_columns         : 21
target_variable     : Churn
identifier_column   : customerID


**What this does:** Formalizes the dataset documentation written earlier into a
structured Python dictionary. **Why this method:** capturing metadata in code (not just
prose) makes it easy to reference or log programmatically later in a real pipeline.
**Insight:** this snapshot is exactly what a teammate (or a future me) would need to
understand this dataset without re-deriving everything from scratch. **ML pipeline
impact:** metadata like this is often stored alongside a dataset (a "data dictionary" or
"data card") in real ML projects, supporting reproducibility and responsible documentation.


---
## 13. Dataset Dimensions

### Step 1 — Understand the Concept
(Recap from Sprint 3) Dataset dimensions refer to its shape — the number of rows
(observations) and columns (variables) — the most basic, essential fact to establish
about any dataset before deeper analysis.

### Step 2 — Demonstrate the Concept
**Business example:** Knowing there are 7,043 customers and 21 columns immediately tells
me the scale of the analysis — enough rows for meaningful statistics, and a manageable
number of columns to inspect individually.

**Why it matters during data analysis:** Dimensions are checked again after every major
cleaning step (dropping duplicates, removing bad rows) to confirm exactly how much data
was affected.

### Step 3 — Implement the Concept


In [14]:
print(f"Dataset dimensions: {df.shape}")
print(f"  Rows (observations): {df.shape[0]:,}")
print(f"  Columns (variables) : {df.shape[1]}")


Dataset dimensions: (7043, 21)
  Rows (observations): 7,043
  Columns (variables) : 21


**What this does:** Reports the dataset's shape. **Why this method:** `.shape` is
the fastest, most direct way to confirm dataset size. **Insight:** 7,043 rows is a solid
sample size for detecting genuine churn patterns, and 21 columns is a manageable number
to inspect thoroughly (done in full in Notebook 2). **ML pipeline impact:** dataset size
directly informs modeling choices later — 7,043 rows is enough for a reasonable
train/test split (Sprint 2, Notebook 7) without needing extreme techniques for small
data.


---
## Summary

| Concept | What it is in THIS dataset |
|---|---|
| Dataset | 7,043 telecom customers, one row each |
| Feature | 19 candidate input columns (services, billing, demographics) |
| Target | `Churn` (Yes/No) |
| Independent Variables | All columns except `customerID` and `Churn` |
| Dependent Variable | `Churn` (same as target, statistics terminology) |
| Numerical Features | `SeniorCitizen`, `tenure`, `MonthlyCharges` (as currently stored) |
| Categorical Features | 16 columns, including `TotalCharges` (currently mis-stored as text) |
| Discrete Feature | `tenure` (whole months) |
| Continuous Feature | `MonthlyCharges` (any decimal value) |
| Identifier Column | `customerID` (unique per row, not predictive) |
| Metadata | Dataset name, source, domain, target, documented above |
| Dataset Dimensions | 7,043 rows x 21 columns |

**Key finding flagged for the next notebook:** `TotalCharges` is stored as text
(`object` dtype), not numeric, despite representing a currency amount — this needs to be
investigated and fixed. This is exactly the kind of real, undocumented data-quality issue
this sprint is designed to teach us to catch.

**Next notebook:** `02_Data_Inspection.ipynb` — Number of Rows/Columns, Data Types,
Column Names, Index, `head()`, `tail()`, `sample()`, `shape`, `info()`, `describe()`,
`dtypes`, `nunique()`, `value_counts()`, and identifying numerical, categorical,
date/time, identifier, and potential target columns.
